
# 02 Smart Beta 与因子投资：核心笔记

> **本章要解决的问题**：同样是买 100 只股票，为什么“怎么加权”能决定赚多赚少？Smart Beta 到底是“聪明的 beta”，还是“披着 beta 外衣的因子暴露”？普通投资者和一人量化，怎么判断一个 smart beta 产品值不值得买、值不值得做？

## 本章导读

这一章是 Ghayur、Heaney 与 Platt《Equity Smart Beta and Factor Investing for Practitioners》的零基础笔记。不用金融背景，只要会一点 Python 和基础统计就能读。

按“是什么 → 有哪些因子 → 怎么捕获 → 怎么组合 → 成本与容量 → 对普通人的启示”展开：

1. Smart Beta 是什么，和被动、主动、因子投资什么关系；
2. 四个常见股票因子：价值、动量、低波、质量；因子收益的三种解释；
3. 捕获因子的三件事：因子定义、加权方案、收益与风险分解；
4. 加权方案实测：市值/等权/基本面/风险/因子打分；
5. 因子捕获回归：组合收益能被因子解释多少；
6. 多因子分散化与组合构建；
7. 成本、容量、跟踪误差与 ESG；
8. 对一人量化的启示：别只看回测收益。

本章数据全部是**合成教学数据**（100 只股票、800 个交易日、4 个因子），参数为教学示例，不代表真实市场。方法的可迁移，数字不能直接拿去投资。

### 学习目标

- 能用自己的话解释 Smart Beta 与被动投资、主动投资、因子投资的区别。
- 能说出价值、动量、低波、质量四个因子分别在抓什么，各举一个常用指标。
- 能解释因子收益的三种来源：风险补偿、行为偏差、结构性摩擦。
- 能比较五种加权方案在收益、跟踪误差、换手、集中度上的差异。
- 能读懂因子捕获回归：alpha、beta、R² 分别说明什么。
- 能估算成本与容量，并说清“为什么不能只看回测收益”。

### 先修

会算平均值、标准差、相关系数；会一点 Python（numpy/pandas 基础）。不需要金融背景。

---

## 在量化研究中的位置

- **谁用**：资产配置/FOF、产品设计、零售投资者、策略研究员。
- **输入**：股票池、因子暴露、加权规则、交易成本与容量假设。
- **输出**：因子组合权重、跟踪误差、成本后收益、因子归因、产品说明书。
- **在流程的哪一步**：在“单因子研究”之后、“组合优化与实盘”之前；它回答“怎么把因子做成一个可投资的产品”。
- **失败模式**：只看回测收益、不写清加权规则、忽略成本/容量、把 Smart Beta 当无风险 alpha。
- **最小工作流**：定义因子 → 选加权 → 测暴露/跟踪误差/换手 → 算成本与容量 → 判断值不值得做。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. Smart Beta 是什么（和不是什么）

### 1.1 一句话定义

**Smart Beta = 用规则化、透明、低成本的方式，获得某类“因子”的暴露。** 它不靠基金经理的主观选股，而是靠一套写死的规则（选股 + 加权）来复制某个因子的收益。

它是“介于被动和主动之间”的产品：比纯被动（市值加权）多了一点“倾斜”，比主动基金便宜、透明、容量大。

### 1.2 三种投资方式对比

| | 被动（纯 beta） | Smart Beta | 主动 |
|---|---|---|---|
| 选股 | 全市场 | 按因子规则筛选 | 基金经理主观判断 |
| 加权 | 市值加权 | 因子打分/等权/风险加权等 | 主观决定 |
| 费用 | 最低 | 低 | 高 |
| 透明度 | 高 | 高（规则公开） | 低 |
| 目标 | 拿市场平均 | 拿因子溢价 | 跑赢基准 |
| 主要风险 | 市场下跌 | 因子失效、风格跑输 | 经理风险、风格漂移 |

### 1.3 为什么机构愿意用

- **收益增强**：长期看，价值、动量等因子有正溢价；
- **风险分散**：不同因子相关性低，可以降低组合波动；
- **低成本**：规则化产品费率远低于主动基金；
- **容量大**：规则透明、可复制，能装下大资金；
- **可解释**：收益能拆成“市场 + 因子 + 残差”，便于向投资人交代。

### 1.4 三个常见误解

1. **“Smart 就是聪明、一定赚钱”**——错。因子会失效，smart beta 也会跑输。
2. **“Smart Beta 是 alpha”**——错。它是**透明的 beta 暴露**，长期收益来自因子溢价，不是经理的选股能力。
3. **“谁都能做”**——规则简单，但成本、容量、换手、跟踪误差决定它能不能真正落地。

---

## 2. 四个常见股票因子与三种解释

### 2.1 四个因子

| 因子 | 抓什么 | 常用指标 | 一句话逻辑 |
|---|---|---|---|
| 价值 | 便宜的股票 | 账面市值比 B/P、盈利收益率 E/P | 便宜的被低估，长期回归 |
| 动量 | 近期强势的股票 | 过去 12 个月（去掉最近 1 个月）收益 | 趋势会延续，投资者反应不足 |
| 低波 | 波动小的股票 | 已实现波动率、下行波动 | 低风险股票长期风险调整后收益更高 |
| 质量 | 好公司 | ROE、毛利率、盈利稳定性、低杠杆 | 好公司更抗跌、盈利更持续 |

### 2.2 因子收益的三种解释

同一个因子溢价，学界有三种解释，理解它们才能判断“这个因子还能不能继续用”：

1. **风险补偿**：因子承担了某种风险，收益是补偿。例如价值股多是困境行业，承担了基本面恶化的风险；
2. **行为偏差**：投资者系统性犯错。例如追涨杀跌（动量）、过度外推（价值反转）、偏爱“彩票型”股票（低波异象）；
3. **结构性摩擦**：制度或资金流造成。例如指数纳入/剔除的被动买卖、融资约束、税收、流动性溢价。

**实践含义**：靠“风险补偿”的因子更稳但可能长期跑输；靠“行为偏差”的因子可能被套利消灭；靠“结构性摩擦”的因子会随制度变化突然失效。

### 2.3 低波动异象：最反直觉的一个

按 CAPM，低波动股票应该收益更低；但现实中低波组合的风险调整后收益往往更高。常见解释：投资者加杠杆受限，只能买高波动股票博收益；机构被基准约束，必须持有高波动股票；散户偏爱彩票型股票。**低波异象提醒我们：风险和收益的关系不是简单的“高风险高收益”。**

---

## 3. 捕获因子：三件事

### 3.1 第一件：因子定义（规格）

同一个“价值因子”，用 B/P 还是 E/P、用整体市场还是行业内排序，结果都不一样。**因子定义就是产品的“配方”**：配方不同，暴露、换手、容量都不同。

### 3.2 第二件：加权方案

选出股票之后，怎么分配权重，直接决定“因子暴露的纯度”：

| 加权 | 做法 | 优点 | 问题 |
|---|---|---|---|
| 市值加权 | 按市值 | 容量大、换手低 | 因子暴露被大市值稀释 |
| 等权 | 每只一样 | 简单、偏小盘 | 小盘流动性差、换手高 |
| 基本面加权 | 按营收/账面值 | 介于市值与等权之间 | 仍可能偏离因子 |
| 风险加权 | 按 1/波动 | 降波动、提夏普 | 低波暴露被放大 |
| 因子打分加权 | 按因子分数 | 因子暴露最纯 | 集中、换手高 |

### 3.3 第三件：收益与风险分解

一个因子组合的收益可以写成：

$$r_{\text{组合}}\approx \underbrace{\beta_{\text{市场}}r_{\text{市场}}}_{\text{市场 beta}}+\underbrace{\sum_k \beta_k f_k}_{\text{因子暴露}}+\underbrace{\alpha}_{\text{残差}}.$$

- **跟踪误差（TE）**：组合收益与基准收益之差的波动，衡量“跑偏程度”；
- **换手**：每年买卖多少比例，决定成本；
- **容量**：策略能装多少钱而不显著推高成本。

### 3.4 代码：100 只股票、4 个因子的合成面板



In [2]:

import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(9802)
N, T = 100, 800                                  # 100 只股票、800 天
names = [f"S{i:03d}" for i in range(N)]

# 1) 基础收益：市场 + 特质（用于计算动量/波动暴露）
mkt = rng.normal(0, 0.010, T)
mkt = mkt - mkt.mean() + 0.0002                  # 只保留 5%/年的目标漂移
beta = rng.normal(1.0, 0.15, N)
idio = rng.normal(0, 0.012, (T, N))
idio = idio - idio.mean()                        # 去均值，避免样本噪声冒充“超额收益”
base_ret = beta[None, :] * mkt[:, None] + idio
base_price = 100 * np.cumprod(1 + base_ret, axis=0)

# 2) 四个可观测特征：B/P、ROE、动量、波动
bp_raw = np.zeros((T, N))
roe_raw = np.zeros((T, N))
bp_raw[0] = rng.normal(0, 0.5, N)
roe_raw[0] = rng.normal(0, 1.0, N)
for t in range(1, T):
    bp_raw[t] = 0.995 * bp_raw[t - 1] + rng.normal(0, 0.06, N)
    roe_raw[t] = 0.995 * roe_raw[t - 1] + rng.normal(0, 0.08, N)
mom_raw = np.zeros((T, N))
vol_raw = np.zeros((T, N))
for t in range(60, T):
    mom_raw[t] = base_price[t - 1] / base_price[t - 61] - 1
    vol_raw[t] = base_ret[t - 60:t].std(axis=0, ddof=1) * np.sqrt(252)


def zs(a):
    s = np.nanstd(a, axis=1, keepdims=True)
    return (a - np.nanmean(a, axis=1, keepdims=True)) / np.where(s > 0, s, 1)


Z = {"价值": zs(bp_raw), "动量": zs(np.nan_to_num(mom_raw)),
     "低波": -zs(np.nan_to_num(vol_raw)), "质量": zs(roe_raw)}

# 3) 收益里加入因子溢价（教学设定），后面的组合都基于这份收益
premia = {"价值": 0.025, "动量": 0.030, "低波": 0.025, "质量": 0.020}
eval_ret = base_ret.copy()
for k, lam in premia.items():
    eval_ret = eval_ret + lam / 252 * Z[k]
valid = np.arange(80, T - 1)

print("面板：", eval_ret.shape, "（天数, 股票数）；合成教学数据")
print("因子暴露相关性：",
      {f"{a}-{b}": round(float(np.corrcoef(Z[a][valid].ravel(), Z[b][valid].ravel())[0, 1]), 2)
       for i, a in enumerate(Z) for b in list(Z)[i + 1:]})

# 4) 因子多空收益：做多分数最高 30 只、做空最低 30 只
def ls_series(Zx):
    out = np.full(T - 1, np.nan)
    for t in valid:
        idx = np.argsort(Zx[t])
        out[t] = eval_ret[t + 1][idx[-30:]].mean() - eval_ret[t + 1][idx[:30]].mean()
    return out


F = pd.DataFrame({k: ls_series(Z[k]) for k in Z}).iloc[80:]
print(f"\n{'因子':<6}{'年化多空':>10}{'年化波动':>10}{'Sharpe':>8}")
for k in Z:
    s = F[k].dropna()
    print(f"{k:<6}{s.mean() * 252:>10.2%}{s.std(ddof=1) * np.sqrt(252):>10.2%}"
          f"{s.mean() / s.std(ddof=1) * np.sqrt(252):>8.2f}")
print("因子之间平均相关性：",
      round(float(F.corr().to_numpy()[np.triu_indices(4, 1)].mean()), 2))


面板： (800, 100) （天数, 股票数）；合成教学数据
因子暴露相关性： {'价值-动量': 0.01, '价值-低波': -0.05, '价值-质量': -0.0, '动量-低波': 0.0, '动量-质量': 0.01, '低波-质量': 0.03}

因子          年化多空      年化波动  Sharpe
价值         5.39%     4.88%    1.11
动量         4.74%     4.99%    0.95
低波         5.74%     6.03%    0.95
质量         4.60%     5.09%    0.90
因子之间平均相关性： 0.0



**读数与应用**：四个因子多空组合年化 4.6%–5.7%，波动 4.9%–6.0%，Sharpe 约 0.9–1.1；因子之间相关性接近 0，这是多因子分散化的基础。**注意：这是合成数据，因子效果被“做”得比较干净；真实市场里因子的 Sharpe 通常低得多、而且会阶段性失效。**

**常见错误**：

1. 把因子多空收益直接当“产品收益”：多空组合需要做空，很多产品只能做多。
2. 因子定义随手换：换一个定义，暴露和收益都会变。
3. 只看收益不看波动：同样 5% 年化，波动 5% 和 15% 是两个产品。

---

## 4. 加权方案实战：同一批股票，不同的“配方”

同样是“价值排名前 30 只”，用不同加权方式，结果会差多少？下面用市值、等权、基本面、风险、因子打分五种方案，与“全市场市值加权”基准对比，重点看四个指标：年化收益、跟踪误差、年换手、有效持股数（1/HHI，越大越分散）。

### 4.1 代码：五种加权方案对比



In [3]:

log_cap = 10 + rng.normal(0, 0.6, N)[None, :] + np.cumsum(rng.normal(0, 0.05, (T, N)), axis=0)
cap = np.exp(log_cap)


def weights_for(t, kind, Zx):
    top = np.argsort(Zx[t])[-30:]
    w = np.zeros(N)
    if kind == "市值":
        w[top] = cap[t, top] / cap[t, top].sum()
    elif kind == "等权":
        w[top] = 1 / 30
    elif kind == "基本面":
        fund = cap[t, top] * np.exp(bp_raw[t, top])
        w[top] = fund / fund.sum()
    elif kind == "风险":
        v = np.nan_to_num(vol_raw[t, top], nan=0.2)
        inv = 1 / np.clip(v, 0.05, None)
        w[top] = inv / inv.sum()
    elif kind == "打分":
        s = np.clip(Zx[t][top], 0, None)
        w[top] = s / s.sum()
    return w


def build_W(kind, Zx, rebal=21):
    W = np.zeros((T, N))
    for t in range(80, T - 1, rebal):
        W[t:t + rebal] = weights_for(t, kind, Zx)
    return W


def perf(W, bench_W, rets):
    pr = np.nansum(W[:-1] * rets[1:], axis=1)[valid]
    br = np.nansum(bench_W[:-1] * rets[1:], axis=1)[valid]
    vol = pr.std(ddof=1) * np.sqrt(252)
    nav = np.cumprod(1 + pr)
    mdd = (nav / np.maximum.accumulate(nav) - 1).min()
    turn = np.abs(np.diff(W, axis=0)).sum(axis=1)[valid].sum() * 252 / len(valid)
    hhi = float(np.sum(W[100] ** 2))             # 某次调仓日的集中度
    return dict(ann=pr.mean() * 252, vol=vol, sharpe=pr.mean() * 252 / vol,
                te=(pr - br).std(ddof=1) * np.sqrt(252), mdd=mdd,
                turnover=turn, neff=1 / hhi if hhi > 0 else np.nan)


bench_W = np.zeros((T, N))                        # 基准：全市场市值加权
for t in range(80, T - 1, 21):
    bench_W[t:t + 21] = cap[t] / cap[t].sum()
b = perf(bench_W, bench_W, eval_ret)
print(f"基准（全市场市值加权）：年化 {b['ann']:.2%}，波动 {b['vol']:.2%}，Sharpe {b['sharpe']:.2f}")
print(f"\n{'方案':<8}{'年化':>8}{'波动':>8}{'Sharpe':>8}{'跟踪误差':>9}{'年换手':>8}{'有效持股':>9}")
for kind in ["市值", "等权", "基本面", "风险", "打分"]:
    r = perf(build_W(kind, Z["价值"]), bench_W, eval_ret)
    print(f"{kind:<8}{r['ann']:>8.2%}{r['vol']:>8.2%}{r['sharpe']:>8.2f}"
          f"{r['te']:>9.2%}{r['turnover']:>8.1f}{r['neff']:>9.1f}")


基准（全市场市值加权）：年化 3.55%，波动 15.25%，Sharpe 0.23

方案            年化      波动  Sharpe     跟踪误差     年换手     有效持股
市值         3.38%  15.98%    0.21    4.77%     6.8     19.9
等权         4.44%  15.58%    0.29    4.08%     5.1     30.0
基本面        4.09%  16.22%    0.25    5.22%     6.3     14.2
风险         4.61%  15.46%    0.30    4.07%     5.6     29.5
打分         5.76%  15.71%    0.37    4.29%     6.0     24.2



**读数与应用**：同样是价值前 30 只，因子打分加权年化 5.76%、市值加权只有 3.38%——差在“因子暴露的纯度”。等权和风险加权更分散（有效持股约 30），但收益低于打分加权；基本面加权集中度最高（有效持股 14），跟踪误差也最大；所有方案年换手 5–7 倍，**这个换手水平对成本很敏感**。基准年化 3.55%，说明“倾斜”确实带来了超额，但代价是 4%–5% 的跟踪误差。

**常见错误**：

1. 只看收益，不看跟踪误差：客户要的是“像基准”，跑偏太大拿不住。
2. 忘记权重归一化与整数手，实盘和回测会有差异。
3. 用月度调仓就以为换手低：成分股变化大时，换手照样很高。

---

## 5. 因子捕获回归：收益到底从哪来

### 5.1 怎么读一张因子捕获表

把组合收益对“市场 + 四个因子”做回归：

$$r_p-r_f=\alpha+\beta_{\text{市场}}(r_m-r_f)+\sum_k\beta_k f_k+\varepsilon.$$

- **alpha**：扣掉市场和因子暴露后，剩下的“真本事”；
- **beta**：组合对每个因子的暴露，因子 beta 越大，收益越依赖该因子；
- **R²**：组合收益的波动有多少能被这些因子解释，越接近 1 越“透明”。

### 5.2 代码：暴露表与捕获回归



In [4]:

# 四因子合成分数组合
W_comp = np.zeros((T, N))
for t in range(80, T - 1, 21):
    score = sum(Z[k][t] for k in Z) / 4
    top = np.argsort(score)[-30:]
    w = np.zeros(N)
    s = np.clip(score[top], 0, None)
    w[top] = s / s.sum()
    W_comp[t:t + 21] = w


def avg_exposure(W):
    return {k: float(np.mean([W[tt] @ Z[k][tt] for tt in valid])) for k in Z}


print("组合平均因子暴露：")
for name, W in [("基准(全市场)", bench_W), ("价值打分", build_W("打分", Z["价值"])),
                ("四因子合成", W_comp)]:
    print(f"  {name:<10}", {k: round(v, 2) for k, v in avg_exposure(W).items()})


def capture(W, label):
    pr = np.nansum(W[:-1] * eval_ret[1:], axis=1)[valid]
    X = np.column_stack([np.ones(len(valid)), mkt[81:81 + len(valid)], F.to_numpy()])
    coef, *_ = np.linalg.lstsq(X, pr, rcond=None)
    resid = pr - X @ coef
    r2 = 1 - resid.var() / pr.var()
    print(f"{label:<10} alpha(年化) {coef[0] * 252:>7.2%}，市场 {coef[1]:>5.2f}，"
          f"价值 {coef[2]:>5.2f}，动量 {coef[3]:>5.2f}，低波 {coef[4]:>5.2f}，"
          f"质量 {coef[5]:>5.2f}，R² {r2:.2f}")


print("\n因子捕获回归（日收益，对市场+四因子）：")
capture(build_W("打分", Z["价值"]), "价值打分")
capture(W_comp, "四因子合成")


组合平均因子暴露：
  基准(全市场)    {'价值': 0.07, '动量': -0.01, '低波': 0.05, '质量': -0.1}
  价值打分       {'价值': 1.3, '动量': -0.01, '低波': -0.1, '质量': 0.03}
  四因子合成      {'价值': 0.59, '动量': 0.59, '低波': 0.51, '质量': 0.74}

因子捕获回归（日收益，对市场+四因子）：
价值打分       alpha(年化)  -0.98%，市场  1.01，价值  0.48，动量 -0.00，低波  0.00，质量 -0.02，R² 0.96
四因子合成      alpha(年化)  -0.81%，市场  1.01，价值  0.21，动量  0.22，低波  0.21，质量  0.23，R² 0.95



**读数与应用**：价值打分组合平均价值暴露 1.30，其他因子接近 0；四因子合成组合四个暴露都在 0.5–0.7 之间。捕获回归里价值组合市场 beta≈1.0、价值 beta 0.48、其他因子 beta≈0，R²=0.96——**96% 的收益波动能被“市场 + 价值因子”解释，alpha 约 −1%，接近 0。** 四因子合成 R²=0.95，四个因子 beta 都在 0.2 左右。这就是 smart beta 的本质：**收益透明、可解释，但几乎没有“神秘 alpha”**。

**常见错误**：

1. 把 R² 高当成“好”：R² 高只说明收益能被解释，不说明收益高。
2. 用日收益回归就宣称 alpha 显著：噪声大时要看标准误和样本长度。
3. 拿不同基准、不同区间的回归结果互相比较。

---

## 6. 多因子分散化与组合构建

### 6.1 单因子 vs 多因子

四个因子相关性接近 0，把它们的组合“混”在一起，可以在不牺牲收益的情况下降低波动——这就是分散化的价值。三种常见混法：

1. **组合等权混合**：四个单因子组合各 25%；
2. **因子打分合成**：把四个 z-score 平均成一个总分，再选股加权；
3. **风险平价**：按每个因子组合的波动倒数分配权重。

### 6.2 代码：三种多因子 blend



In [5]:

W_factors = [build_W("打分", Z[k]) for k in Z]
W_eqblend = sum(W_factors) / 4
vols = {k: F[k].std(ddof=1) for k in Z}
rp_w = {k: (1 / vols[k]) / sum(1 / v for v in vols.values()) for k in Z}
W_rp = sum(rp_w[k] * W_factors[i] for i, k in enumerate(Z))
portfolios = {"单因子-价值": W_factors[0], "四因子-合成": W_comp,
              "四因子-等权混合": W_eqblend, "四因子-风险平价": W_rp}
print(f"{'组合':<16}{'年化':>8}{'波动':>8}{'Sharpe':>8}{'TE':>8}{'换手':>7}{'回撤':>8}")
for name, W in portfolios.items():
    r = perf(W, bench_W, eval_ret)
    print(f"{name:<16}{r['ann']:>8.2%}{r['vol']:>8.2%}{r['sharpe']:>8.2f}"
          f"{r['te']:>8.2%}{r['turnover']:>7.1f}{r['mdd']:>8.2%}")
print("因子间平均相关性:", round(float(F.corr().to_numpy()[np.triu_indices(4, 1)].mean()), 2))


组合                    年化      波动  Sharpe      TE     换手      回撤
单因子-价值             5.76%  15.71%    0.37   4.29%    6.0 -21.56%
四因子-合成             7.88%  14.74%    0.53   4.39%    8.7 -19.92%
四因子-等权混合           6.45%  14.74%    0.44   3.28%    5.8 -19.47%
四因子-风险平价           6.46%  14.79%    0.44   3.28%    5.8 -19.53%
因子间平均相关性: 0.0



**读数与应用**：单因子价值组合年化 5.76%、波动 15.7%；四因子合成把年化提到 7.88%、波动降到 14.7%，Sharpe 从 0.37 升到 0.53。等权混合与风险平价结果接近（因为四个因子波动差不多，风险平价权重≈1/4），Sharpe 0.44。**多因子的好处是“同等风险下更高收益”，不是“保证更高收益”。**

**常见错误**：

1. 把相关性为 0 当“没有共同风险”：危机时因子相关性会上升。
2. 多因子简单相加，导致某个因子权重过大。
3. 忽略换手：四因子合成的年换手 8.7，比单因子高近一半。

---

## 7. 成本、容量与 ESG

### 7.1 成本：换手的“价格”

年换手 6 倍、单边成本 30bp，一年成本约 $6\times30\text{bp}=180\text{bp}=1.8\%$。对 Sharpe 0.5 的策略，这不是小数目。**回测收益和实盘收益之间，差的就是成本。**

### 7.2 容量：能装多少钱

规模越大，交易占市场成交量的比例越高，冲击成本越大。粗略用平方根模型：$\text{冲击}\approx c\,\sigma\sqrt{Q/V}$。**容量不是“能募多少钱”，而是“在可接受成本下能装多少钱”。**

### 7.3 ESG 与 Smart Beta

ESG 因子与 smart beta 可以叠加：在因子选股后再做 ESG 筛选，或者把 ESG 当第五个因子。但要注意：**叠加筛选会降低分散度、提高跟踪误差**，而且 ESG 数据质量参差、口径不统一。

### 7.4 代码：成本与容量



In [6]:

print("成本后 Sharpe（30bp/单边换手）：")
for name, W in portfolios.items():
    r = perf(W, bench_W, eval_ret)
    net = r["sharpe"] - r["turnover"] * 0.0030 / r["vol"]
    print(f"  {name:<16} 毛 {r['sharpe']:.2f} → 净 {net:.2f}（年换手 {r['turnover']:.1f}）")

print("\n容量（成本后 Sharpe；单票 ADV 3 亿元、冲击系数 0.5）：")
for name, W in portfolios.items():
    r = perf(W, bench_W, eval_ret)
    row = []
    for aum in [1e9, 5e9, 2e10]:
        part = r["turnover"] / 252 * aum / (N * 3e8)
        extra_bps = 0.5 * 0.02 * np.sqrt(max(part, 1e-8)) * 1e4
        net = r["ann"] - r["turnover"] * extra_bps / 1e4
        row.append(net / r["vol"])
    print(f"  {name:<16} 10亿 {row[0]:.2f} | 50亿 {row[1]:.2f} | 200亿 {row[2]:.2f}")

print("\n一张选择表（教学示例，数字取决于假设）：")
print(f"{'候选':<16}{'年化':>8}{'Sharpe':>8}{'TE':>8}{'净Sharpe':>9}{'200亿':>8}{'结论':>6}")
for name, W in portfolios.items():
    r = perf(W, bench_W, eval_ret)
    net = r["sharpe"] - r["turnover"] * 0.0030 / r["vol"]
    cap200 = r["ann"] - r["turnover"] * (0.5 * 0.02 * np.sqrt(
        max(r["turnover"] / 252 * 2e10 / (N * 3e8), 1e-8)) * 1e4) / 1e4
    verdict = "可做" if net > 0.3 and r["te"] < 0.06 else "观望"
    print(f"{name:<16}{r['ann']:>8.2%}{r['sharpe']:>8.2f}{r['te']:>8.2%}"
          f"{net:>9.2f}{cap200 / r['vol']:>8.2f}{verdict:>6}")


成本后 Sharpe（30bp/单边换手）：
  单因子-价值           毛 0.37 → 净 0.25（年换手 6.0）
  四因子-合成           毛 0.53 → 净 0.36（年换手 8.7）
  四因子-等权混合         毛 0.44 → 净 0.32（年换手 5.8）
  四因子-风险平价         毛 0.44 → 净 0.32（年换手 5.8）

容量（成本后 Sharpe；单票 ADV 3 亿元、冲击系数 0.5）：
  单因子-价值           10亿 0.36 | 50亿 0.34 | 200亿 0.32
  四因子-合成           10亿 0.51 | 50亿 0.49 | 200亿 0.44
  四因子-等权混合         10亿 0.43 | 50亿 0.41 | 200亿 0.39
  四因子-风险平价         10亿 0.43 | 50亿 0.41 | 200亿 0.39

一张选择表（教学示例，数字取决于假设）：
候选                    年化  Sharpe      TE  净Sharpe    200亿    结论
单因子-价值             5.76%    0.37   4.29%     0.25    0.32    观望
四因子-合成             7.88%    0.53   4.39%     0.36    0.44    可做
四因子-等权混合           6.45%    0.44   3.28%     0.32    0.39    可做
四因子-风险平价           6.46%    0.44   3.28%     0.32    0.39    可做



**读数与应用**：30bp 成本下，单因子价值组合净 Sharpe 0.37→0.25，四因子合成 0.53→0.36；规模升到 200 亿，合成组合从 0.51 降到 0.44。选择表里“可做”的标准是净 Sharpe>0.3 且跟踪误差<6%，这是**假设示例**，真实产品要按自己的资金、成本和约束重算。

**常见错误**：

1. 成本只算佣金，不算冲击和价差。
2. 用回测的 ADV 算容量：自己的交易本身就是 ADV 的一部分。
3. ESG 筛选后不复核跟踪误差与行业暴露。

---

## 8. 对一人量化的启示

### 8.1 四句话

1. **Smart Beta 不是“聪明钱”**，它是规则化的因子暴露；赚的是因子溢价，不是神秘 alpha。
2. **加权方案和因子定义一样重要**：同一批股票，打分加权和市值加权是两种产品。
3. **回测收益要打折**：成本、容量、跟踪误差、换手，任何一项都能把“漂亮曲线”打成平庸。
4. **因子会失效**：拥挤、估值、制度变化都会杀死溢价，要持续监控。

### 8.2 一张检查清单

- [ ] 因子定义写清楚了吗？换了定义结果会怎样？
- [ ] 加权方案说清楚了吗？集中度、换手多少？
- [ ] 跟踪误差多大？客户/自己能承受吗？
- [ ] 成本后 Sharpe 还有多少？最坏成本假设下呢？
- [ ] 容量多大？自己的钱占 ADV 多少？
- [ ] 因子暴露是多少？alpha 是真本事还是暴露？
- [ ] 因子相关性会不会在危机时上升？

---

## 本章小结

### 一张表

| 概念 | 一句话 | 用途 |
|---|---|---|
| Smart Beta | 规则化的因子暴露 | 低成本拿因子溢价 |
| 四个因子 | 价值/动量/低波/质量 | 收益来源 |
| 三种解释 | 风险补偿/行为偏差/结构摩擦 | 判断因子可持续性 |
| 加权方案 | 市值/等权/基本面/风险/打分 | 决定暴露纯度与容量 |
| 跟踪误差 | 相对基准的跑偏程度 | 产品是否“像基准” |
| 因子捕获 | alpha + beta + R² | 收益分解与归因 |
| 多因子 blend | 分散化降波动 | 提升风险调整收益 |
| 成本/容量 | 换手×成本、规模×冲击 | 决定能否落地 |

### 八句话

1. Smart Beta 是“透明的因子 beta”，不是 alpha。
2. 价值、动量、低波、质量是四个常见股票因子。
3. 因子收益来自风险补偿、行为偏差或结构性摩擦。
4. 因子定义和加权方案，决定暴露的“纯度”。
5. 市值加权的容量最大，但因子暴露会被稀释。
6. 因子捕获回归里，R² 高说明收益“透明”，不说明收益高。
7. 多因子的价值在“同等风险下更高收益”，不是保证赚钱。
8. 不看成本与容量，回测收益只是纸面数字。

---

## 本章习题

**1.** 用一句话说明 Smart Beta 与被动投资、主动投资的区别。

**2.** 价值、动量、低波、质量四个因子分别抓什么？各举一个常用指标。

**3.** 因子收益的三种解释是什么？各举一个例子。

**4.** 同一批股票，市值加权、等权、因子打分加权，为什么收益和跟踪误差不同？

**5.** 年换手 6 倍、单边成本 30bp，一年成本大约多少？对 Sharpe 0.5 的策略影响多大？

**6.** 因子捕获回归结果：alpha −1%、市场 beta 1.0、价值 beta 0.48、R² 0.96。怎么读？

**7.** 低波动异象和 CAPM 矛盾在哪？有哪些常见解释？

**8.** 为什么说“一人量化别只看回测收益”？列出至少三个必须看的指标。

<details><summary>参考答案</summary>

**1.** 被动投资按市值买全市场、拿市场平均；主动投资靠主观选股争取超额；Smart Beta 用公开规则（选股+加权）获得某类因子暴露，费用和透明度接近被动、目标接近因子溢价。

**2.** 价值抓“便宜”（B/P、E/P）；动量抓“近期强势”（12-1 个月收益）；低波抓“低风险”（已实现波动率）；质量抓“好公司”（ROE、毛利率、低杠杆）。

**3.** 风险补偿（价值股承担困境风险）、行为偏差（追涨杀跌、过度外推、偏爱彩票型股票）、结构性摩擦（指数纳入剔除、融资约束、税收、流动性）。

**4.** 加权改变每只股票的权重，从而改变组合的因子暴露纯度、集中度、换手和容量：打分加权暴露最纯但集中、换手高；市值加权容量大但暴露被大市值稀释；等权偏小盘、更分散。

**5.** 成本约 $6\times30\text{bp}=180\text{bp}=1.8\%$/年。对波动 15% 的组合，约吃掉 $1.8/15\approx0.12$ 的 Sharpe；若毛 Sharpe 只有 0.5，成本后约 0.38。

**6.** 市场 beta≈1，说明承担了市场风险；价值 beta 0.48，说明约一半的主动收益来自价值因子暴露；alpha≈−1%，扣掉因子后没有正超额；R²=0.96，说明收益波动几乎全部能被市场和价值因子解释——典型的 Smart Beta 组合。

**7.** CAPM 认为高 beta 应高收益，但低波组合风险调整后收益更高。解释：杠杆约束（想加杠杆买高波资产受限）、基准约束（机构必须持有高波股票）、彩票偏好（散户偏爱高波“彩票”）。

**8.** 成本后收益、跟踪误差、换手、容量（规模对冲击/收益的影响）、因子暴露与 alpha、因子相关性/拥挤度、样本外与不同市场环境的表现。

</details>

---

## 参考资料

- Ghayur, K., Heaney, R., & Platt, S. *Equity Smart Beta and Factor Investing for Practitioners*. Wiley, 2019.（原书）
- Ang, A. *Asset Management: A Systematic Approach to Factor Investing*.
- Asness, C., Moskowitz, T., & Pedersen, L. H. *Value and Momentum Everywhere*.
- 本系列 `01_事件驱动回测与撮合`、`04_风险模型与收益归因`；主线第 29、30、39 章。
